# YgoMaster-PackBuilder — YGOPRODeck Parser Tests

Notebook offline per testare `parse_custom_pack(...)`.

Copre parsing valido, metadata incoerenti, campi mancanti, YDK ID non validi,
ordine delle carte e campi esterni ignorati.


## 1. Ambiente e root repository

In [1]:
from pathlib import Path
import sys

print("Python interpreter:", sys.executable)
print("Current working directory:", Path.cwd())

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Impossibile individuare la root della repository.")

print("Repository root:", repo_root)

Python interpreter: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\.venv\Scripts\python.exe
Current working directory: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\tests
Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder


## 2. Import

In [2]:
from copy import deepcopy

from ygomaster_packbuilder.sources.ygoprodeck import (
    YgoProDeckPackCard,
    YgoProDeckPackData,
    parse_custom_pack,
)

print("Imports OK")

Imports OK


## 3. Payload valido

In [3]:
raw_pack = [
    {
        "username": "livius03",
        "pack_name": "Scaled Invasion",
        "pack_description": "A pack centered around Venom and Alien cards.",
        "img_url": "8400623",
        "code": 501,
        "name": '"A" Cell Breeding Device',
        "type": "Continuous Spell",
        "id": 34541863,
        "pack_views": 0,
        "card_rarity": "Common",
    },
    {
        "username": "livius03",
        "pack_name": "Scaled Invasion",
        "pack_description": "A pack centered around Venom and Alien cards.",
        "img_url": "8400623",
        "code": 501,
        "name": "Alien Ammonite",
        "type": "Tuner Monster",
        "id": 91123920,
        "pack_views": 0,
        "card_rarity": "Rare",
    },
]

print("Rows:", len(raw_pack))

Rows: 2


## 4. Parsing valido

In [4]:
pack = parse_custom_pack("24116", raw_pack)

assert isinstance(pack, YgoProDeckPackData)
assert pack.external_id == "24116"
assert pack.name == "Scaled Invasion"
assert pack.creator == "livius03"
assert pack.image_id == "8400623"
assert len(pack.cards) == 2

print(pack)

YgoProDeckPackData(external_id='24116', name='Scaled Invasion', description='A pack centered around Venom and Alien cards.', creator='livius03', image_id='8400623', cards=[YgoProDeckPackCard(name='"A" Cell Breeding Device', ydk_id=34541863, card_type='Continuous Spell', source_rarity='Common'), YgoProDeckPackCard(name='Alien Ammonite', ydk_id=91123920, card_type='Tuner Monster', source_rarity='Rare')])


## 5. Carte parsate

In [5]:
first = pack.cards[0]
second = pack.cards[1]

assert isinstance(first, YgoProDeckPackCard)
assert first.name == '"A" Cell Breeding Device'
assert first.ydk_id == 34541863
assert first.card_type == "Continuous Spell"
assert first.source_rarity == "Common"

assert second.name == "Alien Ammonite"
assert second.ydk_id == 91123920
assert second.source_rarity == "Rare"

print("Cards OK")

Cards OK


## 6. ID numerico come stringa

In [6]:
case = deepcopy(raw_pack)
case[0]["id"] = "34541863"

parsed = parse_custom_pack("24116", case)

assert parsed.cards[0].ydk_id == 34541863
assert isinstance(parsed.cards[0].ydk_id, int)

print("String ID converted correctly")

String ID converted correctly


## 7. Risposta vuota

In [7]:
try:
    parse_custom_pack("24116", [])
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("Il parser avrebbe dovuto rifiutare una risposta vuota.")

Expected ValueError: YGOPRODeck custom pack '24116' returned no cards.


## 8. `pack_name` incoerente

In [8]:
case = deepcopy(raw_pack)
case[1]["pack_name"] = "Another Pack"

try:
    parse_custom_pack("24116", case)
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("Metadata incoerenti non rilevati.")

Expected ValueError: Inconsistent YGOPRODeck pack metadata at row 1: field 'pack_name' differs from the first row.


## 9. `username` incoerente

In [9]:
case = deepcopy(raw_pack)
case[1]["username"] = "another-user"

try:
    parse_custom_pack("24116", case)
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("Username incoerente non rilevato.")

Expected ValueError: Inconsistent YGOPRODeck pack metadata at row 1: field 'username' differs from the first row.


## 10. `img_url` incoerente

In [10]:
case = deepcopy(raw_pack)
case[1]["img_url"] = "999"

try:
    parse_custom_pack("24116", case)
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("img_url incoerente non rilevato.")

Expected ValueError: Inconsistent YGOPRODeck pack metadata at row 1: field 'img_url' differs from the first row.


## 11. Campo carta mancante

In [11]:
case = deepcopy(raw_pack)
del case[0]["card_rarity"]

try:
    parse_custom_pack("24116", case)
except (ValueError, KeyError) as exc:
    print("Expected parsing error:", type(exc).__name__, "-", exc)
else:
    raise AssertionError("Campo card_rarity mancante non rilevato.")

Expected parsing error: ValueError - Missing field 'card_rarity' in YGOPRODeck response row 0.


## 12. Campo metadata mancante

In [12]:
case = deepcopy(raw_pack)
del case[0]["pack_name"]

try:
    parse_custom_pack("24116", case)
except (ValueError, KeyError) as exc:
    print("Expected parsing error:", type(exc).__name__, "-", exc)
else:
    raise AssertionError("Campo pack_name mancante non rilevato.")

Expected parsing error: ValueError - Missing field 'pack_name' in YGOPRODeck response row 0.


## 13. YDK ID non numerico

In [13]:
case = deepcopy(raw_pack)
case[0]["id"] = "not-a-number"

try:
    parse_custom_pack("24116", case)
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("YDK ID non numerico non rilevato.")

Expected ValueError: Invalid YDK ID at row 0: 'not-a-number'


## 14. Ordine preservato

In [14]:
parsed = parse_custom_pack("24116", raw_pack)

assert [c.name for c in parsed.cards] == [row["name"] for row in raw_pack]

print([c.name for c in parsed.cards])

['"A" Cell Breeding Device', 'Alien Ammonite']


## 15. Campi esterni inutilizzati ignorati

In [15]:
parsed = parse_custom_pack("24116", raw_pack)
card = parsed.cards[0]

assert not hasattr(card, "code")
assert not hasattr(card, "pack_views")

print("Unused fields ignored correctly")

Unused fields ignored correctly
